In [1]:
import pandas as pd
import numpy as np

In [2]:
test = pd.read_csv("testing.csv")
test

,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11425,http://www.fontspace.com/category/blackletter,45,17,0,2,0,0,0,0,0,...,0,0,0,448,5396,3980,0,0,6,legitimate
11426,http://www.budgetbots.com/server.php/Server%20...,84,18,0,5,0,1,1,0,0,...,1,0,0,211,6728,0,0,1,0,phishing
11427,https://www.facebook.com/Interactive-Televisio...,105,16,1,2,6,0,1,0,0,...,0,0,0,2809,8515,8,0,1,10,legitimate
11428,http://www.mypublicdomainpictures.com/,38,30,0,2,0,0,0,0,0,...,1,0,0,85,2836,2455493,0,0,4,legitimate


In [3]:
test['label'] = test['status'].apply(lambda x: 1 if x == 'legitimate' else 0)

In [4]:
test['NoOfSelfRef'] = (test['ratio_intHyperlinks'] * test['nb_hyperlinks']).round().astype(int)

In [5]:
test.rename(columns={
    "length_url":"URLLength",
    "domain_with_copyright":"HasCopyrightInfo",
    "http_in_path":"IsHTTPS"
}, inplace=True)

In [15]:
social_sites = ['facebook', 'twitter', 'instagram', 'linkedin', 'youtube', 'pinterest', 'tiktok']

def has_social_net(url):
    if isinstance(url, str):
        return int(any(site in url.lower() for site in social_sites))
    else:
        return 0
test['HasSocialNet'] = test['url'].apply(has_social_net)

In [10]:
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

def has_description(url):
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            description = soup.find('meta', attrs={'name': 'description'})
            return int(description is not None)
        else:
            return 0
    except:
        return 0

def process_urls_in_parallel(urls, max_workers=20):
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {executor.submit(has_description, url): url for url in urls}
        for future in as_completed(future_to_url):
            url = future_to_url[future]
            try:
                result = future.result()
            except Exception as e:
                result = 0
            results.append(result)
    return results

urls = test['url'].tolist()  

print("⏳ Starting scraping...")
description_flags = process_urls_in_parallel(urls, max_workers=20)
test['HasDescription'] = description_flags

print("✅ Scraping Done!")

⏳ Starting scraping...


C:\Users\soman\AppData\Local\Temp\ipykernel_2444\3665465914.py:14: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.text, 'html.parser')


✅ Scraping Done!


In [21]:
colstosave = ["HasSocialNet", "IsHTTPS", "HasCopyrightInfo", "NoOfSelfRef", "HasDescription", "URLLength", "label"]

In [22]:
df_selected = test[colstosave]

In [23]:
df_selected.to_csv('testing_urls.csv', index=False)